# Economy of Bits — Notebook 07: the compute stack over thirty years

Notebook 06 finished the hypothesis tests. The divergence is **hyperscalers**, not large
firms (+0.0139, *p* = 0.0003, against tail-vs-everyone at *p* = 0.27). The bridge is
**negative** — capital spending does not predict share gains. And the layers move in three
distinct directions, all at *p* < 0.0001:

| | capex 2016 → 2025 | asset lightness | share gain |
|---|---|---|---|
| designs compute | 0.071 → **0.032** | 0.889 → **0.932** | **+10.07 pts** |
| operates compute | 0.093 → **0.387** | 0.837 → **0.616** | +9.79 pts |
| fabricates compute | 0.100 → 0.154 | 0.807 → 0.749 | +0.25 pts |

Two things are still outstanding.

**The classification is wrong at the edges.** Analog Devices, NXP, Skyworks and Qorvo sit in
"designs compute" and all four *own fabs*. Their capital intensity (0.053–0.083) is double
the true fabless group's (0.010–0.039). Cell 3 replaces sector intuition with one rule —
**who owns the fab** — applied identically to every firm, and checkable line by line.

**The classification was written after seeing the result.** It needs out-of-sample evidence,
and the panel runs to 1996. Twenty years before the AI build-out did not inform it in any
way. The arc is stark: Intel went from 5.19% of the top hundred firms by value to 0.38%,
while fabless designers went from 0.17% to 14.67%.

Upload `bits_inputs_v7.zip`. No downloads; runtime about two minutes.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# Notebook 06 finished the hypothesis tests. Three results, all clean:
#
#   DIVERGENCE   hyperscalers vs other tail firms, capex extra trend +0.0139,
#                p = 0.0003. But TAIL vs everyone else is +0.0038, p = 0.27.
#                It is not "large firms". It is the firms that operate compute.
#   BRIDGE       capital spending does NOT predict gains in value share:
#                -0.115 (p = 0.067) in the panel, -0.435 (p = 0.19) in the
#                cross-section. Both negative.
#   LAYERS       designs compute +10.07 points of index value on mean capex
#                0.040; operates compute +9.79 points on 0.158; fabricates
#                compute +0.25 points on 0.124.
#
# The layer trends are three distinct directions, all at p < 0.0001:
#
#     designs      capex 0.071 -> 0.032   asset lightness 0.889 -> 0.932
#     operates     capex 0.093 -> 0.387   asset lightness 0.837 -> 0.616
#     fabricates   capex 0.100 -> 0.154   asset lightness 0.807 -> 0.749
#
# This notebook does the two things still outstanding.
#
# FIRST, the classification is wrong at the edges and it matters. Analog Devices,
# NXP, Skyworks and Qorvo are in "designs compute" and all four OWN FABS -- they
# are integrated device manufacturers, and their capital intensity (0.053 to
# 0.083) is twice the true fabless group's (0.010 to 0.039). The rule should be
# fab ownership, not sector.
#
# SECOND, and more important: the layer classification was written after seeing
# the result, so it needs out-of-sample evidence. The panel runs to 1996. Twenty
# years before the AI build-out is real out-of-sample, and the arc is stark:
# Intel went from 5.19% of the top hundred firms by value to 0.38%, while
# fabless designers went from 0.17% to 14.67%. If that holds up under scrutiny
# it is a thirty-year natural experiment inside one index, and it is the paper.
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import stats as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

RUN_TAG = "bits_07_stack"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, CACHE, FIGS = ROOT / "outputs", ROOT / "cache", ROOT / "outputs" / "figures"
for d in (OUTD, CACHE, FIGS):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet"); df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name); print(f"  saved {name}  ({len(df):,} rows)")

def trend(s, maxlags=3):
    s = s.dropna()
    if len(s) < 5:
        return np.nan, np.nan
    X = sm.add_constant(s.index.values.astype(float) - np.mean(s.index.values))
    f = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return float(f.params[1]), float(f.pvalues[1])

print(f"writing to: {ROOT}")

## 2 · Inputs, including the full panel back to 1996

In [ ]:
# =============================================================================
# CELL 2 - Inputs, including the FULL panel back to 1996
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v7.zip"), recursive=True)):
        dest = CACHE / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}"); return dest
    for d in (BASE / "inputs7", BASE, CACHE / "inputs"):
        if (pathlib.Path(d) / "sp500_marketcap_panel_clean.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v7.zip not found.")

full = pd.read_csv(SRC / "sp500_marketcap_panel_clean.csv.gz")
full["year"] = full.year.astype(int)
F = pd.read_csv(SRC / "firm_measures_v3.csv"); F["year"] = F.year.astype(int)
rosters = pd.read_csv(SRC / "sp500_membership_reconstructed.csv")
pub_ineq = pd.read_csv(SRC / "inequality_by_year.csv")
ROSTER = {int(y): set(g.ticker) for y, g in rosters.groupby("year")}
print(f"full panel  {len(full):,} firm-years, {full.year.min()}-{full.year.max()}, "
      f"{full.ticker.nunique()} tickers")
print(f"measured    {len(F):,} firm-years, 2016-2025")

# Coverage is the governing constraint on anything said about the early years.
cov = []
for y in sorted(full.year.unique()):
    got = set(full[full.year == y].ticker); ros = ROSTER.get(y, set())
    cov.append({"year": y, "roster": len(ros), "in_panel": len(got),
                "coverage": len(got) / len(ros) if ros else np.nan})
coverage = pd.DataFrame(cov)
print("\ncoverage of the roster, by decade:")
print(coverage.assign(dec=(coverage.year // 5) * 5).groupby("dec")
      .agg(years=("year", "count"), mean_coverage=("coverage", "mean")).round(3).to_string())
print("\n40% in 1996 rising to 97% in 2025. Any share computed on the whole panel")
print("has a moving denominator, so the long view in cell 4 is normalised on the")
print("TOP 100 FIRMS BY VALUE, which is close to fully covered even in 1996 --")
print("the panel holds 187 firms that year and the largest are the ones that")
print("survive to be quoted today.")
savetable(coverage, "coverage_1996_2025")

## 3 · The taxonomy, on one rule: who owns the fab?

AMD is classified as *fabricates* through 2008 and *designs* from 2009, when it spun its
fabs out as GlobalFoundries. Not special pleading — the rule applied to a firm that changed
sides, which gives cell 6 a within-firm test.

In [ ]:
# =============================================================================
# CELL 3 - The taxonomy, on one rule: WHO OWNS THE FAB?
#
# Notebook 06 classified by sector intuition and got the edges wrong. The rule
# here is a single economic question -- does the firm own the plant that turns
# its design into a physical object? -- and it is applied the same way to every
# firm. It is checkable line by line, which sector intuition is not.
#
#   DESIGNS    owns no fab. The product is a design or a piece of software, and
#              reproducing it costs almost nothing. Fabless chip firms and EDA.
#   FABRICATES owns fabs. Integrated device manufacturers.
#   EQUIPMENT  sells the machines that fill the fabs.
#   OPERATES   runs hyperscale data centres. Buys the chips, owns the buildings.
#
# AMD is the interesting case and is handled explicitly: it owned fabs until it
# spun them out as GlobalFoundries in March 2009, and it is classified as
# FABRICATES through 2008 and DESIGNS from 2009. That is not a special pleading,
# it is the rule applied to a firm that changed sides -- and it gives a
# within-firm event that cell 6 uses.
# =============================================================================
DESIGNS = ["NVDA", "AVGO", "QCOM", "MRVL", "XLNX", "ALTR", "BRCM", "LSI",
           "SNPS", "CDNS", "ANET"]
FABRICATES = ["INTC", "TXN", "MU", "ADI", "MCHP", "ON", "NXPI", "SWKS", "QRVO",
              "MPWR", "NSM", "LLTC", "MXIM", "FCS", "IDTI", "CY", "ATML"]
EQUIPMENT = ["AMAT", "LRCX", "KLAC", "TER", "NVLS", "KLIC", "ENTG"]
OPERATES = ["MSFT", "GOOGL", "GOOG", "AMZN", "META", "FB", "ORCL"]

def assign_layer(ticker, year, gics_digital):
    t = str(ticker).upper()
    if t == "AMD":                      # fabs spun out as GlobalFoundries, 2009
        return "fabricates compute" if year <= 2008 else "designs compute"
    if t in DESIGNS:
        return "designs compute"
    if t in FABRICATES:
        return "fabricates compute"
    if t in EQUIPMENT:
        return "compute equipment"
    if t in OPERATES:
        return "operates compute"
    return "other digital" if gics_digital else "not digital"

DIGITAL_SECTORS = {"Information Technology", "Communication Services"}
DIGITAL_EXTRA = {"AMZN", "TSLA", "NFLX", "BKNG", "EBAY", "ABNB", "UBER", "DASH", "SHOP"}
for df in (full, F):
    df["gics_digital"] = (df.sector.isin(DIGITAL_SECTORS)
                          | df.ticker.str.upper().isin(DIGITAL_EXTRA))
    df["layer"] = [assign_layer(t, y, g) for t, y, g
                   in zip(df.ticker, df.year, df.gics_digital)]

COMPUTE = ["designs compute", "fabricates compute", "compute equipment",
           "operates compute"]
print("firms per layer, measured window:")
print(F.groupby("layer").ticker.nunique().to_string())
print("\nfirms per compute layer across the FULL panel, with their years:")
for L in COMPUTE:
    sub = full[full.layer == L]
    spans = sub.groupby("ticker").year.agg(["min", "max"])
    print(f"\n  {L}:")
    print("    " + ", ".join(f"{t} ({r['min']}-{r['max']})" for t, r in spans.iterrows()))

# Does the corrected rule separate on capital intensity the way it should?
print("\ncapital intensity by layer under the corrected rule (2016-2025 mean):")
chk = (F[F.layer.isin(COMPUTE)].groupby("layer")
       .agg(firms=("ticker", "nunique"), mean_capex=("capex_intensity", "mean"),
            mean_asset_light=("asset_light", "mean")).round(3))
display(chk)
print("The four analog firms moved out of DESIGNS should widen the gap between")
print("designs and fabricates. If designs is not clearly the lightest layer, the")
print("rule has not done its job and the classification needs another look.")
savetable(F[["year", "ticker", "firm", "layer", "capex_intensity", "asset_light",
             "cap_share", "mktcap"]], "firm_layers_measured")

## 4 · Thirty years of the compute stack

The out-of-sample test. Normalised on the top 100 by value, because roster coverage moves
from 40% to 97% and a moving denominator would manufacture a trend.

In [ ]:
# =============================================================================
# CELL 4 - Thirty years of the compute stack
#
# This is the out-of-sample test the layer hypothesis needs. It was formed on
# 2016-2025 data. The panel runs to 1996, and the two decades before the AI
# build-out did not inform the classification in any way.
#
# Normalised on the top 100 firms by value rather than the whole panel, because
# coverage moves from 40% to 97% across the period and a moving denominator
# would manufacture a trend. The largest hundred are close to fully covered even
# in 1996.
# =============================================================================
top100 = (full.sort_values(["year", "mktcap"], ascending=[True, False])
          .groupby("year").head(100))
tot = top100.groupby("year").mktcap.sum()
share = (100 * top100.groupby(["year", "layer"]).mktcap.sum()
         .unstack().div(tot, axis=0))
counts = top100.groupby(["year", "layer"]).size().unstack()

print("each layer's share of the top 100 firms by value (%):")
display(share[[c for c in COMPUTE if c in share.columns]].round(2))
print("\nnumber of firms from each layer in the top 100:")
display(counts[[c for c in COMPUTE if c in counts.columns]].fillna(0).astype(int))

rows = []
for L in COMPUTE:
    if L not in share.columns:
        continue
    s = share[L].dropna()
    for lo, hi, lab in [(1996, 2015, "1996-2015, before"),
                        (2016, 2025, "2016-2025, measured"),
                        (1996, 2025, "1996-2025, all")]:
        w = s[(s.index >= lo) & (s.index <= hi)]
        if len(w) < 5:
            continue
        b, p = trend(w)
        rows.append({"layer": L, "window": lab, "first": round(float(w.iloc[0]), 2),
                     "last": round(float(w.iloc[-1]), 2), "slope_pp_per_year": round(b, 4),
                     "p": round(p, 4)})
long_trends = pd.DataFrame(rows)
print("\ntrends in each layer's share, before and during the measured window:")
display(long_trends)
savetable(long_trends, "layer_share_trends_30y")
savetable(share.reset_index(), "layer_share_top100_30y")

print("\n" + "=" * 72)
print("THE CASE OF INTEL")
print("=" * 72)
print("One firm makes the argument concrete. Intel owned the fabs, and for most")
print("of the period it also designed the best chips. It is the fabricating")
print("layer in a single company.")
for t in ["INTC", "NVDA", "TXN", "MU"]:
    s = top100[top100.ticker == t].set_index("year").mktcap / tot
    s = (100 * s).dropna()
    if len(s):
        print(f"  {t:<5} {s.iloc[0]:5.2f}% ({s.index[0]}) -> {s.iloc[-1]:5.2f}% "
              f"({s.index[-1]})   peak {s.max():.2f}% in {int(s.idxmax())}")
print("\nIf Intel's share falls by an order of magnitude while Nvidia's rises by")
print("more, across thirty years and through three technology cycles, then the")
print("relationship between owning the physical plant and capturing the value is")
print("not a fact about the AI build-out. It is the standing arrangement, and")
print("2023-2025 is its most extreme expression.")

## 5 · Decomposing the first paper's result

The JRFM paper said the top 1% went from 11.3% to 30.0% of index value. It never said which
firms supplied the 18.7 points. This is the arithmetic that joins the two papers.

In [ ]:
# =============================================================================
# CELL 5 - Decomposing the first paper's result
#
# The JRFM paper reported that the top 1% share of index value rose from 11.3%
# to 30.0%, a gain of 18.7 points. It did not say which firms supplied the gain.
# This is the arithmetic that joins the two papers: an exact decomposition, no
# model, no estimation. Each layer's contribution is its change in share of
# index value, and the contributions sum to the total by construction.
# =============================================================================
est = full[full.year.between(2016, 2025)].copy()
est["cap_share"] = est.mktcap / est.groupby("year").mktcap.transform("sum")
lay_share = (100 * est.groupby(["year", "layer"]).cap_share.sum().unstack())
print("each layer's share of index value, 2016-2025 (%):")
display(lay_share.round(2))

first, last = lay_share.iloc[0], lay_share.iloc[-1]
dec = pd.DataFrame({"share_2016": first.round(2), "share_2025": last.round(2),
                    "change_pp": (last - first).round(2)})
dec["share_of_total_movement"] = (100 * dec.change_pp / dec.change_pp.abs().sum() * 2).round(1)
dec = dec.sort_values("change_pp", ascending=False)
print("\nDECOMPOSITION -- where the concentration came from:")
display(dec)
savetable(dec.reset_index(), "concentration_decomposition")

compute_gain = dec.loc[[L for L in COMPUTE if L in dec.index], "change_pp"].sum()
print(f"\nthe four compute layers together: {compute_gain:+.2f} points of index value")
print(f"everything else: {dec.change_pp.sum() - compute_gain:+.2f}")
print("(the column sums to zero by construction -- shares of a whole)")

# The same question asked of the actual top 1%, which is what the first paper
# measured, rather than of the whole distribution.
print("\nwho IS the top 1%, year by year?")
rows = []
for y in range(2016, 2026):
    s = est[est.year == y]
    cut = s.cap_share.quantile(0.99)
    t = s[s.cap_share >= cut].sort_values("cap_share", ascending=False)
    rows.append({"year": y, "n": len(t), "share_pct": round(100 * t.cap_share.sum(), 1),
                 "firms": ", ".join(t.ticker),
                 "compute_firms": ", ".join(t.loc[t.layer.isin(COMPUTE), "ticker"])})
who = pd.DataFrame(rows)
display(who)
savetable(who, "top1_membership_by_year")
print("\nIf the top 1% is entirely compute-layer firms by 2025 and was not in")
print("2016, the first paper's headline and this paper's mechanism are the same")
print("event described at two levels of resolution.")

## 6 · Capital efficiency by layer, and the AMD case

In [ ]:
# =============================================================================
# CELL 6 - Two tests the corrected taxonomy makes possible
#
# TEST ONE, value capture per unit of capital. The layers differ in both what
# they earn and what they spend, and the ratio is the economically interesting
# quantity: how much index value did a layer gain for each point of revenue it
# ploughed into plant?
#
# TEST TWO, AMD. It owned fabs until 2009 and has been fabless since. The rule
# in cell 3 moves it between layers at that date, which makes it a within-firm
# test of the whole argument -- the same company, the same products, a different
# position in the stack. If the layer story is real, AMD's capital intensity
# should fall and its value share should behave like a designer afterwards.
# =============================================================================
lay = (F.dropna(subset=["cap_share"]).groupby(["ticker", "layer"], as_index=False)
       .agg(firm=("firm", "first"), years=("year", "nunique"),
            first_share=("cap_share", "first"), last_share=("cap_share", "last"),
            mean_capex=("capex_intensity", "mean")))
lay = lay[lay.years >= 8].copy()
lay["d_log_share"] = np.log(lay.last_share / lay.first_share)
summ = (lay.groupby("layer").agg(
            n=("ticker", "nunique"), median_gain=("d_log_share", "median"),
            mean_capex=("mean_capex", "mean")).round(3))
summ["gain_per_capex_point"] = (summ.median_gain / summ.mean_capex).round(2)
print("value captured per unit of capital committed:")
display(summ.sort_values("gain_per_capex_point", ascending=False))
savetable(summ.reset_index(), "capital_efficiency_by_layer")
print("\nThis ratio is descriptive, not causal, and the paper must say so. It")
print("says how the decade turned out, not what would have happened had a firm")
print("chosen differently.")

print("\n" + "=" * 72)
print("AMD, BEFORE AND AFTER IT SOLD ITS FABS")
print("=" * 72)
amd = full[full.ticker == "AMD"].sort_values("year").copy()
amd["cap_share_pct"] = 100 * amd.mktcap / amd.year.map(
    full.groupby("year").mktcap.sum())
amd["era"] = np.where(amd.year <= 2008, "owns fabs (to 2008)", "fabless (2009 on)")
display(amd.groupby("era").agg(years=("year", "count"),
                               mean_share_pct=("cap_share_pct", "mean"),
                               first_year=("year", "min"),
                               last_year=("year", "max")).round(3))
amd_m = F[F.ticker == "AMD"]
if len(amd_m):
    print("\nAMD capital intensity in the measured window (fabless throughout):")
    print(amd_m.set_index("year")[["capex_intensity", "asset_light"]].round(3).to_string())
print("\nAMD's share of index value, by year:")
print(amd.set_index("year").cap_share_pct.round(3).to_string())
savetable(amd[["year", "ticker", "mktcap", "cap_share_pct", "era", "layer"]], "amd_event")
print("\nThe honest reading: the fab sale happened in the middle of a financial")
print("crisis and AMD's recovery took another five years, so this is an")
print("illustration rather than an identified effect. It belongs in the paper as")
print("a case, clearly labelled, not as evidence.")

## 7 · Figures and the results file

In [ ]:
# =============================================================================
# CELL 7 - Figures, and the results file the paper is written from
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, ACC, GREY, GRN = "#1f3b57", "#c8702a", "#7a7a7a", "#4a7c59"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
ax = axes[0]
style = {"designs compute": (INK, "-"), "operates compute": (ACC, "-"),
         "fabricates compute": (GRN, "--"), "compute equipment": (GREY, ":")}
for L, (c, ls) in style.items():
    if L in share.columns:
        s = share[L].dropna()
        ax.plot(s.index, s.values, ls, color=c, lw=2, label=L.replace(" compute", ""))
ax.axvspan(2016, 2025, color=INK, alpha=0.06, lw=0)
ax.annotate("measured\nwindow", (2020.5, ax.get_ylim()[1] * 0.9), ha="center",
            fontsize=7.5, color=GREY)
ax.set_xlabel("year"); ax.set_ylabel("% of top-100 market value")
ax.set_title("Thirty years of the compute stack"); ax.legend(fontsize=7.5)

ax = axes[1]
for L, (c, ls) in style.items():
    d = F[F.layer == L]
    if not len(d):
        continue
    s = d.groupby("year").capex_intensity.mean().dropna()
    ax.plot(s.index, s.values, ls, color=c, lw=2, label=L.replace(" compute", ""))
ax.set_xlabel("year"); ax.set_ylabel("capex / revenue")
ax.set_title("Who carries the capital"); ax.legend(fontsize=7.5)

ax = axes[2]
dd = dec.drop(index=[i for i in dec.index if i not in list(COMPUTE) +
                     ["other digital", "not digital"]], errors="ignore")
cols = [INK if i in COMPUTE else GREY for i in dd.index]
ax.barh(range(len(dd)), dd.change_pp.values, color=cols, height=0.6)
ax.set_yticks(range(len(dd)))
ax.set_yticklabels([i.replace(" compute", "") for i in dd.index], fontsize=8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("change in share of index value, 2016-2025 (pp)")
ax.set_title("Where the concentration came from")
fig.suptitle("Notebook 07: the economy of bits moved up one layer",
             fontsize=10, color=GREY, y=1.02)
savefig(fig, "S1_stack")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
rows = []
for p in files:
    row = {"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size / 1024, 1)}
    if p.suffix == ".parquet":
        try:
            row["rows"] = len(pd.read_parquet(p))
        except Exception:
            row["rows"] = None
    rows.append(row)
manifest = pd.DataFrame(rows); manifest.to_csv(OUTD / "MANIFEST.csv", index=False)
display(manifest)
(OUTD / "run_summary.json").write_text(json.dumps(
    {"run": RUN_TAG, "when": datetime.datetime.now().isoformat(timespec="seconds"),
     "layers": {L: sorted(full.loc[full.layer == L, "ticker"].unique().tolist())
                for L in COMPUTE},
     "tables": saved_tables}, indent=2, default=str))

print("\n" + "=" * 70)
print("SEND ME")
print("=" * 70)
print("  layer_share_trends_30y.csv        the out-of-sample test")
print("  layer_share_top100_30y.csv        the thirty-year series")
print("  concentration_decomposition.csv   where the first paper's 18.7 points came from")
print("  top1_membership_by_year.csv       who the top 1% actually is")
print("  capital_efficiency_by_layer.csv")
print()
print("If the designs-versus-fabricates divergence is already visible before")
print("2016, the empirical work is finished. The next thing I send is a draft.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive); print("Download started.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")